# Day 4: Next-Match Targets and Leakage-Safe Feature Engineering

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objective
Create chronological batting and bowling datasets with next-match regression targets. Features for a player-match row must use information available before that match. The target comes from that player's next eligible match.

## Inputs
- `data/processed/batting_player_match.csv`
- `data/processed/bowling_player_match.csv`
- `data/processed/match_metadata.csv`

If your Day 3 output uses a different filename or schema, update the configuration cell only after inspecting the actual file and columns.

## Targets
- Batting: `next_match_runs`
- Bowling: `next_match_wickets`, `next_match_runs_conceded`

## Rules
- Sort each player's records chronologically before shifting targets.
- Do not use current-match statistics as predictors for the target on the same row.
- Feature rolling windows use only prior rows (`shift(1)` before rolling).
- Preserve a chronological holdout; do not randomly split rows.
- This notebook creates target/feature tables only. It does not train models.


In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

# Locate project root even when Jupyter is opened from the notebooks/ folder.
current_dir = Path.cwd().resolve()
PROJECT_ROOT = None

for candidate in [current_dir, *current_dir.parents]:
    if (candidate / "data" / "processed").is_dir() or (
        (candidate / "data" / "raw").is_dir() and (candidate / "notebooks").is_dir()
    ):
        PROJECT_ROOT = candidate
        break

# Fallback: set this to your actual project root if auto-detection fails.
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\\Git\\Personal\\ipl-player-performance")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

BATTING_INPUT = PROCESSED_DIR / "batting_player_match.csv"
BOWLING_INPUT = PROCESSED_DIR / "bowling_player_match.csv"
MATCH_INPUT = PROCESSED_DIR / "match_metadata.csv"

print("Current working directory:", current_dir)
print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED_DIR)
for file_path in [BATTING_INPUT, BOWLING_INPUT, MATCH_INPUT]:
    print(f"{file_path.name}: {'FOUND' if file_path.exists() else 'MISSING'}")


Current working directory: D:\Git\Personal\ipl-player-performance\notebooks
Project root: D:\Git\Personal\ipl-player-performance
Processed directory: D:\Git\Personal\ipl-player-performance\data\processed
batting_player_match.csv: FOUND
bowling_player_match.csv: FOUND
match_metadata.csv: FOUND


In [2]:
def read_required_csv(path: Path, required_columns: set[str]) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Complete and validate Day 3 first, "
            "then update the input filename if your Day 3 output differs."
        )
    df = pd.read_csv(path)
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(
            f"{path.name} is missing required columns: {sorted(missing)}. "
            f"Available columns: {list(df.columns)}"
        )
    return df

# Required minimum schema; optional columns are inspected in the next cell.
batting = read_required_csv(
    BATTING_INPUT, {"match_id", "player_name", "date", "runs"}
)
bowling = read_required_csv(
    BOWLING_INPUT, {"match_id", "player_name", "date", "wickets", "runs_conceded"}
)
matches = read_required_csv(MATCH_INPUT, {"match_id", "date"})

for name, df in [("batting", batting), ("bowling", bowling), ("matches", matches)]:
    print(f"\\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
    print("Columns:", list(df.columns))


\nbatting: 18,768 rows x 24 columns
Columns: ['match_id', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'match_type', 'team', 'opposition', 'player_name', 'player_id', 'player_name_norm', 'team_norm', 'opposition_norm', 'runs', 'balls_faced', 'fours', 'sixes', 'batting_position', 'was_dismissed', 'dismissal_kind', 'innings_count', 'strike_rate']
\nbowling: 14,700 rows x 24 columns
Columns: ['match_id', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'match_type', 'team', 'opposition', 'player_name', 'player_id', 'player_name_norm', 'team_norm', 'opposition_norm', 'legal_balls', 'runs_conceded', 'wickets', 'wides', 'noballs', 'dots', 'innings_count', 'overs', 'economy']
\nmatches: 1,243 rows x 9 columns
Columns: ['match_id', 'source_file', 'date', 'season', 'venue', 'city', 'team_1', 'team_2', 'match_type']


## 1. Inspect schemas and date quality

Review these outputs before continuing. Do not guess a column mapping if the notebook reports missing required columns. The source may use `player` instead of `player_name`, or a different date representation; inspect the CSV and change the mapping explicitly if needed.


In [3]:
def normalize_dates(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    invalid = int(out["date"].isna().sum())
    print(f"{name}: invalid/missing dates = {invalid:,}")
    if invalid:
        display(out.loc[out["date"].isna()].head(10))
        raise ValueError(f"{name} contains invalid dates. Resolve before target creation.")
    return out

batting = normalize_dates(batting, "batting")
bowling = normalize_dates(bowling, "bowling")
matches = normalize_dates(matches, "match metadata")

for name, df in [("batting", batting), ("bowling", bowling)]:
    duplicate_key = df.duplicated(["match_id", "player_name"], keep=False)
    print(f"{name}: duplicated match/player keys = {int(duplicate_key.sum()):,}")
    if duplicate_key.any():
        display(df.loc[duplicate_key].sort_values(["match_id", "player_name"]).head(20))
        raise ValueError(
            f"{name} must have one row per match_id + player_name before Day 4. "
            "Resolve Day 3 duplicates; do not aggregate blindly."
        )


batting: invalid/missing dates = 0
bowling: invalid/missing dates = 0
match metadata: invalid/missing dates = 0
batting: duplicated match/player keys = 0
bowling: duplicated match/player keys = 0


## 2. Validate match order and join metadata

A match may share a date with another match. Use match ID as a deterministic secondary sort key, but inspect IDs if same-day ordering matters. The player sequence is ordered by date then match ID; no random ordering is used.


In [4]:
# Avoid creating conflicting date columns during metadata join.
metadata_cols = ["match_id", "date"]
for optional in ["season", "venue", "city", "team_1", "team_2", "team1", "team2"]:
    if optional in matches.columns:
        metadata_cols.append(optional)

match_meta = matches[metadata_cols].drop_duplicates()
if match_meta["match_id"].duplicated().any():
    bad = match_meta.loc[match_meta["match_id"].duplicated(keep=False)]
    display(bad.head(20))
    raise ValueError("Match metadata has multiple rows per match_id.")

def attach_metadata(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    if "date" in out.columns:
        out = out.rename(columns={"date": "player_match_date"})
    merged = out.merge(
        match_meta.rename(columns={"date": "match_date"}),
        on="match_id", how="left", validate="many_to_one", suffixes=("", "_meta")
    )
    if merged["match_date"].isna().any():
        missing = merged.loc[merged["match_date"].isna(), ["match_id", "player_name"]].drop_duplicates()
        display(missing.head(20))
        raise ValueError(f"{name}: rows lack a matching date in match metadata.")
    # Prefer metadata date as the canonical match date.
    if "player_match_date" in merged.columns:
        discrepancy = merged["player_match_date"].dt.date != merged["match_date"].dt.date
        print(f"{name}: player dataset vs metadata date discrepancies = {int(discrepancy.sum()):,}")
        if discrepancy.any():
            display(merged.loc[discrepancy, ["match_id", "player_name", "player_match_date", "match_date"]].head(10))
            raise ValueError(f"{name}: date discrepancy detected; resolve before target generation.")
    merged["date"] = merged["match_date"]
    return merged.drop(columns=["player_match_date"], errors="ignore")

batting = attach_metadata(batting, "batting")
bowling = attach_metadata(bowling, "bowling")


batting: player dataset vs metadata date discrepancies = 0
bowling: player dataset vs metadata date discrepancies = 0


## 3. Create next-match targets

The `shift(-1)` target is attached to the current row, while predictor features are created only from earlier matches. The final row for each player has no known next-match target and is excluded from supervised target tables.


In [5]:
def add_next_match_target(
    df: pd.DataFrame,
    group_col: str,
    target_map: dict[str, str],
    dataset_name: str,
) -> pd.DataFrame:
    out = df.copy()
    out = out.sort_values([group_col, "date", "match_id"], kind="stable").reset_index(drop=True)
    grouped = out.groupby(group_col, sort=False, dropna=False)
    for source_col, target_col in target_map.items():
        if source_col not in out.columns:
            raise ValueError(
                f"{dataset_name}: target source column {source_col!r} is missing. "
                f"Available columns: {list(out.columns)}"
            )
        out[target_col] = grouped[source_col].shift(-1)
    out["next_match_id"] = grouped["match_id"].shift(-1)
    out["next_match_date"] = grouped["date"].shift(-1)
    out["matches_seen_before"] = grouped.cumcount()
    return out

batting_targets_all = add_next_match_target(
    batting, "player_name", {"runs": "next_match_runs"}, "batting"
)
bowling_targets_all = add_next_match_target(
    bowling, "player_name",
    {"wickets": "next_match_wickets", "runs_conceded": "next_match_runs_conceded"},
    "bowling"
)

batting_targets = batting_targets_all.dropna(subset=["next_match_runs"]).copy()
bowling_targets = bowling_targets_all.dropna(
    subset=["next_match_wickets", "next_match_runs_conceded"]
).copy()

print("Batting rows with target:", len(batting_targets))
print("Bowling rows with targets:", len(bowling_targets))
print("Batting rows without next match:", len(batting_targets_all) - len(batting_targets))
print("Bowling rows without next match:", len(bowling_targets_all) - len(bowling_targets))


Batting rows with target: 18030
Bowling rows with targets: 14123
Batting rows without next match: 738
Bowling rows without next match: 577


## 4. Leakage-safe historical features

Rolling and cumulative features are based on prior matches only. Current-match stats remain in the dataset for audit/target alignment but must not be selected as predictors for that same row. These features are calculated from each player's ordered history.


In [6]:
def add_history_features(
    df: pd.DataFrame,
    group_col: str,
    stat_columns: list[str],
    prefix: str,
) -> pd.DataFrame:
    out = df.sort_values([group_col, "date", "match_id"], kind="stable").copy()
    grouped = out.groupby(group_col, sort=False, dropna=False)
    for col in stat_columns:
        if col not in out.columns:
            print(f"Skipping unavailable feature source: {col}")
            continue

        # shift(1) excludes the current match from all history features.
        prior = grouped[col].transform(lambda s: s.shift(1))
        out[f"prior_career_{prefix}_{col}"] = (
            prior.groupby(out[group_col], sort=False, dropna=False)
            .transform(lambda s: s.expanding(min_periods=1).mean())
        )
        out[f"prior_last5_mean_{prefix}_{col}"] = (
            out.groupby(group_col, sort=False, dropna=False)[col]
            .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
        )
        out[f"prior_last5_count_{prefix}_{col}"] = (
            out.groupby(group_col, sort=False, dropna=False)[col]
            .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).count())
        )
    return out

batting_stat_candidates = [
    "runs", "balls_faced", "fours", "sixes", "strike_rate", "dismissed"
]
bowling_stat_candidates = [
    "wickets", "runs_conceded", "legal_balls", "maidens", "dots", "economy"
]

batting_features_all = add_history_features(
    batting_targets_all, "player_name", batting_stat_candidates, "batting"
)
bowling_features_all = add_history_features(
    bowling_targets_all, "player_name", bowling_stat_candidates, "bowling"
)

# Keep rows where the future target exists for supervised training datasets.
batting_features = batting_features_all.dropna(subset=["next_match_runs"]).copy()
bowling_features = bowling_features_all.dropna(
    subset=["next_match_wickets", "next_match_runs_conceded"]
).copy()

display(batting_features.head())
display(bowling_features.head())


Skipping unavailable feature source: dismissed
Skipping unavailable feature source: maidens


,match_id,season,venue,city,team_1,team_2,match_type,team,opposition,player_name,...,prior_last5_count_batting_balls_faced,prior_career_batting_fours,prior_last5_mean_batting_fours,prior_last5_count_batting_fours,prior_career_batting_sixes,prior_last5_mean_batting_sixes,prior_last5_count_batting_sixes,prior_career_batting_strike_rate,prior_last5_mean_batting_strike_rate,prior_last5_count_batting_strike_rate
0,548346,2012,Wankhede Stadium,Mumbai,Mumbai Indians,Deccan Chargers,NaN,Deccan Chargers,Mumbai Indians,A Ashish Reddy,...,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0
1,548352,2012,"MA Chidambaram Stadium, Chepauk",Chennai,Chennai Super Kings,Deccan Chargers,NaN,Deccan Chargers,Chennai Super Kings,A Ashish Reddy,...,1.0,0.000000,0.000000,1.0,1.000000,1.000000,1.0,100.0,100.0,1.0
2,548359,2012,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Deccan Chargers,Kings XI Punjab,NaN,Deccan Chargers,Kings XI Punjab,A Ashish Reddy,...,2.0,0.000000,0.000000,2.0,0.500000,0.500000,2.0,100.0,100.0,2.0
3,548373,2012,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Deccan Chargers,Rajasthan Royals,NaN,Deccan Chargers,Rajasthan Royals,A Ashish Reddy,...,3.0,0.333333,0.333333,3.0,0.333333,0.333333,3.0,100.0,100.0,3.0
4,548376,2012,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Deccan Chargers,Royal Challengers Bangalore,NaN,Deccan Chargers,Royal Challengers Bangalore,A Ashish Reddy,...,4.0,0.750000,0.750000,4.0,0.250000,0.250000,4.0,137.5,137.5,4.0


,match_id,season,venue,city,team_1,team_2,match_type,team,opposition,player_name,...,prior_last5_count_bowling_runs_conceded,prior_career_bowling_legal_balls,prior_last5_mean_bowling_legal_balls,prior_last5_count_bowling_legal_balls,prior_career_bowling_dots,prior_last5_mean_bowling_dots,prior_last5_count_bowling_dots,prior_career_bowling_economy,prior_last5_mean_bowling_economy,prior_last5_count_bowling_economy
0,548341,2012,Subrata Roy Sahara Stadium,Pune,Pune Warriors,Deccan Chargers,NaN,Deccan Chargers,Pune Warriors,A Ashish Reddy,...,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0
1,548346,2012,Wankhede Stadium,Mumbai,Mumbai Indians,Deccan Chargers,NaN,Deccan Chargers,Mumbai Indians,A Ashish Reddy,...,1.0,24.000000,24.000000,1.0,6.000000,6.000000,1.0,8.000000,8.000000,1.0
2,548348,2012,Barabati Stadium,Cuttack,Deccan Chargers,Pune Warriors,NaN,Deccan Chargers,Pune Warriors,A Ashish Reddy,...,2.0,18.500000,18.500000,2.0,7.000000,7.000000,2.0,6.538462,6.538462,2.0
3,548352,2012,"MA Chidambaram Stadium, Chepauk",Chennai,Chennai Super Kings,Deccan Chargers,NaN,Deccan Chargers,Chennai Super Kings,A Ashish Reddy,...,3.0,18.333333,18.333333,3.0,5.666667,5.666667,3.0,7.914530,7.914530,3.0
4,548356,2012,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bangalore,Deccan Chargers,NaN,Deccan Chargers,Royal Challengers Bangalore,A Ashish Reddy,...,4.0,16.750000,16.750000,4.0,5.000000,5.000000,4.0,7.935897,7.935897,4.0


## 5. Validate target alignment and leakage safeguards

The checks below verify target values against the next chronological record for each player. Any failure stops the notebook. The checks do not prove that every domain-specific scorecard rule is correct; they test sequence alignment and feature timing.


In [7]:
def validate_next_match_targets(
    all_rows: pd.DataFrame,
    group_col: str,
    target_pairs: dict[str, str],
    dataset_name: str,
) -> None:
    ordered = all_rows.sort_values([group_col, "date", "match_id"], kind="stable").copy()
    g = ordered.groupby(group_col, sort=False, dropna=False)
    for source_col, target_col in target_pairs.items():
        expected = g[source_col].shift(-1)
        actual = ordered[target_col]
        comparable = expected.notna() & actual.notna()
        if not np.allclose(
            expected.loc[comparable].astype(float),
            actual.loc[comparable].astype(float),
            equal_nan=True,
        ):
            raise AssertionError(f"{dataset_name}: {target_col} is misaligned.")
    # Every available history feature must be based on earlier records.
    first_rows = ordered.groupby(group_col, sort=False, dropna=False).head(1)
    history_cols = [c for c in ordered.columns if c.startswith("prior_")]
    if history_cols and first_rows[history_cols].notna().any().any():
        raise AssertionError(f"{dataset_name}: first player row has non-null prior-history features.")
    print(f"{dataset_name}: target alignment and first-row history checks passed.")

validate_next_match_targets(
    batting_targets_all, "player_name", {"runs": "next_match_runs"}, "batting"
)
validate_next_match_targets(
    bowling_targets_all, "player_name",
    {"wickets": "next_match_wickets", "runs_conceded": "next_match_runs_conceded"},
    "bowling"
)

for name, df in [("batting", batting_features), ("bowling", bowling_features)]:
    if df["next_match_date"].isna().any():
        raise AssertionError(f"{name}: target rows have missing next_match_date.")
    future_before_current = df["next_match_date"] < df["date"]
    if future_before_current.any():
        display(df.loc[future_before_current, ["player_name", "date", "next_match_date"]].head())
        raise AssertionError(f"{name}: target date precedes feature row date.")
    print(f"{name}: rows={len(df):,}; unique players={df['player_name'].nunique():,}; target date order OK")


batting: target alignment and first-row history checks passed.
bowling: target alignment and first-row history checks passed.
batting: rows=18,030; unique players=633; target date order OK
bowling: rows=14,123; unique players=503; target date order OK


## 6. Save processed target tables

The output files are intended for the next modelling stage. The holdout split is not created here; it will be made chronologically during modelling. Files are saved only after validation cells pass.


In [8]:
BATTING_TARGET_OUT = PROCESSED_DIR / "batting_next_match_features.csv"
BOWLING_TARGET_OUT = PROCESSED_DIR / "bowling_next_match_features.csv"

batting_features.to_csv(BATTING_TARGET_OUT, index=False)
bowling_features.to_csv(BOWLING_TARGET_OUT, index=False)

print("Saved:", BATTING_TARGET_OUT, batting_features.shape)
print("Saved:", BOWLING_TARGET_OUT, bowling_features.shape)


Saved: D:\Git\Personal\ipl-player-performance\data\processed\batting_next_match_features.csv (18030, 49)
Saved: D:\Git\Personal\ipl-player-performance\data\processed\bowling_next_match_features.csv (14123, 50)


In [9]:
day4_text = f"""# Day 4: Next-Match Targets and Feature Engineering

## Outputs
- `data/processed/batting_next_match_features.csv`
- `data/processed/bowling_next_match_features.csv`

## Target definitions
- Batting: `next_match_runs`, derived from the player's next chronological batting record.
- Bowling: `next_match_wickets` and `next_match_runs_conceded`, derived from the player's next chronological bowling record.

## Feature policy
- Player records sorted by date and match ID.
- Rolling means and cumulative averages use `shift(1)` so current-match performance is not included in historical features.
- The final record per player has no next-match target and is excluded from supervised output tables.
- Do not use current-match performance columns as predictors for the target on the same row.

## Validation
- Required input columns checked.
- Dates parsed and validated.
- Duplicate match/player keys checked.
- Match metadata join validated as many-to-one.
- Next-match target alignment checked.
- First player record has no prior-history values.
- Target date must not precede feature-row date.

## Row counts
- Batting input rows: {len(batting):,}
- Batting supervised rows: {len(batting_features):,}
- Batting unique players: {batting_features['player_name'].nunique():,}
- Bowling input rows: {len(bowling):,}
- Bowling supervised rows: {len(bowling_features):,}
- Bowling unique players: {bowling_features['player_name'].nunique():,}

## Limitations and next checks
1. Confirm target semantics with the capstone supervisor.
2. Check same-day doubleheaders and match ordering where relevant.
3. Inspect missing/zero performance statistics and scorecard reconciliation.
4. Build chronological train/validation/test splits on the training days.
5. Compare Linear Regression, Random Forest, and XGBoost using MAE, RMSE, and R².
6. LSTM/GRU sequence modelling using the last five chronological matches is a later step.
7. No model is trained on Day 4.
"""

day4_doc_path = DOCS_DIR / "day4_targets_feature_engineering.md"
day4_doc_path.write_text(day4_text, encoding="utf-8")
print("Saved:", day4_doc_path)


Saved: D:\Git\Personal\ipl-player-performance\docs\day4_targets_feature_engineering.md
